# Transformer Strategy Compare: Diffusion vs Flow Matching

这份 notebook 在同样的 PointNet + DiT backbone 下，对比：
- Flow Matching
- Diffusion

默认把训练轮数设得比主 notebook 短一些，适合作为策略差异对照入口。


In [ ]:
from pathlib import Path

from autodl_unplug_charger_transformer_fm import ExperimentConfig, train_experiment

CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = None
for cand in CANDIDATES:
    if (cand / 'pyproject.toml').exists() and (cand / 'configs').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root. Please open the notebook from the repo root or its notebooks/ directory.')


In [ ]:
# User choices
COMPARE_EPOCHS = 200
STRATEGIES = ['fm', 'diffusion']
RUN_PREFIX = 'unplug_charger_transformer_compare_v1'

N_OBS_STEPS = 3
N_PRED_STEPS = 32
BATCH_SIZE = 64
TRAIN_GRAD_ACCUM_STEPS = 2

HIDDEN_DIM = 512
TIME_DIM = 256
NUM_BLOCKS = 6
NHEAD = 8
DIM_FEEDFORWARD = 2048
DROPOUT = 0.1

LEARNING_RATE = 1e-4
BETAS = (0.9, 0.95)
TRANSFORMER_WEIGHT_DECAY = 1e-3
OBS_ENCODER_WEIGHT_DECAY = 1e-6
LR_WARMUP_STEPS = 1000

SUCCESS_SELECTION_EVERY_EPOCHS = 50
CHECKPOINT_EVERY_EPOCHS = 50
SUCCESS_SELECTION_EPISODES = 10
STANDARD_EVAL_EPISODES = 0

WANDB_ENABLE = True
WANDB_PROJECT = 'pfp-autodl-transformer-fm'
WANDB_MODE = 'online'

# 24G fallback preset:
# BATCH_SIZE = 32
# TRAIN_GRAD_ACCUM_STEPS = 4


In [ ]:
base_cfg = ExperimentConfig(
    train_epochs=COMPARE_EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=TRAIN_GRAD_ACCUM_STEPS,
    n_obs_steps=N_OBS_STEPS,
    n_pred_steps=N_PRED_STEPS,
    hidden_dim=HIDDEN_DIM,
    time_dim=TIME_DIM,
    num_blocks=NUM_BLOCKS,
    nhead=NHEAD,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    learning_rate=LEARNING_RATE,
    betas=BETAS,
    transformer_weight_decay=TRANSFORMER_WEIGHT_DECAY,
    obs_encoder_weight_decay=OBS_ENCODER_WEIGHT_DECAY,
    lr_warmup_steps=LR_WARMUP_STEPS,
    success_selection_every_epochs=SUCCESS_SELECTION_EVERY_EPOCHS,
    checkpoint_every_epochs=CHECKPOINT_EVERY_EPOCHS,
    success_selection_episodes=SUCCESS_SELECTION_EPISODES,
    standard_eval_episodes=STANDARD_EVAL_EPISODES,
    wandb_enable=WANDB_ENABLE,
    wandb_project=WANDB_PROJECT,
    wandb_mode=WANDB_MODE,
)
base_cfg.validate()
{
    'compare_epochs': base_cfg.train_epochs,
    'strategies': STRATEGIES,
    'n_obs_steps': base_cfg.n_obs_steps,
    'batch_size': base_cfg.batch_size,
    'effective_batch_size': base_cfg.batch_size * base_cfg.grad_accum_steps,
    'hidden_dim': base_cfg.hidden_dim,
    'num_blocks': base_cfg.num_blocks,
}


In [ ]:
summaries = {}
for strategy in STRATEGIES:
    run_cfg = ExperimentConfig(**{**base_cfg.__dict__, 'run_name': f'{RUN_PREFIX}_{strategy}'})
    summaries[strategy] = train_experiment(run_cfg, strategy=strategy)

comparison_rows = []
for strategy, summary in summaries.items():
    comparison_rows.append({
        'strategy': strategy,
        'best_metric': summary.get('best_metric'),
        'best_epoch': summary.get('best_epoch'),
        'best_success_rate': summary.get('best_success_rate'),
        'best_success_epoch': summary.get('best_success_epoch'),
        'train_loss_last': summary.get('train_loss_last'),
        'valid_loss_last': summary.get('valid_loss_last'),
    })
comparison_rows
